# Tarea 2: Ruteo Consistente de Vehículos
Integrantes:


In [ ]:
# Imports
import math
import random
from pathlib import Path

In [ ]:

# Rutas relativas a la carpeta desde la que se ejecuta el notebook.
PATH_SOLOMON = Path("data/originales")
PATH_GENERADAS = Path("data/instancias")

ARCHIVO_R108 = PATH_SOLOMON / "R108.txt"
ARCHIVO_R110 = PATH_SOLOMON / "R110.txt"

# Comprobar que existen los originales.
for archivo in [ARCHIVO_R108, ARCHIVO_R110]:
    if not archivo.is_file():
        raise FileNotFoundError(
            f"No se encontró el archivo: {archivo.resolve()}"
        )

print("Entrada:", PATH_SOLOMON.resolve())
print("Salida:", PATH_GENERADAS.resolve())

# Sección 1 Lectura de archivos originales

In [ ]:
def leer_solomon(path):
    """Lee cabecera VEHICLE y filas CUSTOMER, sin depender de espacios exactos."""
    lines = path.read_text(encoding="utf-8-sig").splitlines()
    vehicles = capacity = None
    rows = {}
    in_customers = False
    for line in lines:
        s = line.split()
        if not s:
            continue
        if s[0].upper() == "CUSTOMER":
            in_customers = True
            continue
        if not in_customers and len(s) == 2:
            try:
                vehicles, capacity = int(s[0]), float(s[1])
            except ValueError:
                pass
        if in_customers and len(s) == 7:
            try:
                row = tuple(float(x) for x in s)
            except ValueError:
                continue
            i = int(row[0])
            if i != row[0] or i in rows:
                raise ValueError("Identificador no entero o duplicado")
            if row[3] < 0 or row[4] > row[5] or row[6] < 0:
                raise ValueError(f"Datos invalidos en cliente {i}")
            rows[i] = row
    if 0 not in rows or vehicles is None or capacity is None:
        raise ValueError(f"Formato Solomon incompleto: {path}")
    return rows, vehicles, capacity


def distancia(rows, i, j):
    return math.hypot(
        rows[i][1] - rows[j][1], 
        rows[i][2] - rows[j][2]
        )

# Sección 2 Generación de instancias propias

In [ ]:
def generar(source, destination, n, days, vehicles, seed, slack):
    rows, original_vehicles, original_q = leer_solomon(source)
    ids = sorted(i for i in rows if i != 0)[:n] # Solo los primeros n clientes, sin el depot (0).
    # Validaciones de parametros 
    if len(ids) != n or n < 15 or days < 3 or not 3 <= vehicles <= n:
        raise ValueError("Se requieren >=15 clientes, >=3 dias y 3<=vehiculos<=clientes")
    if not 0 <= slack <= 1:
        raise ValueError("El margen de capacidad debe estar entre 0 y 1")

    # Generacion de demanda diaria aleatoria, con al menos un dia activo por cliente.    
    rng = random.Random(seed)
    demand = {}
    for i in ids:
        active = set(rng.sample(range(days), rng.choice([days - 1, days])))
        demand[i] = [max(1, round(rows[i][3] * rng.uniform(0.8, 1.2)))
                     if t in active else 0 for t in range(days)]

    # Agrupacion geografica sencilla para construir UNA solución factible de referencia.
    ordered = sorted(ids, key=lambda i: math.atan2(
        rows[i][2] - rows[0][2], 
        rows[i][1] - rows[0][1]
        ))

    # Equilibrio de numero de clientes     
    groups = [
        ordered[r*n//vehicles:(r+1)*n//vehicles]
        for r in range(vehicles)
    ]

    # Calculo de cargas diarias para cada grupo.
    loads = [
        [sum(demand[i][t] for i in group) for t in range(days)]
             for group in groups
    ]
    # Toma la mayor carga y se redondea hacia arriba con el margen de slack para obtener la nueva capacidad Q.
    q = math.ceil(max(max(x) for x in loads) * (1 + slack))


    # Preparar horarios de apertura y cierre, y ventanas de servicio para cada cliente.
    opening, original_close = rows[0][4:6]
    service = {i: max(1.0, rows[i][6]) for i in ids}
    ready = {i: rows[i][4] for i in ids}
    due = {i: rows[i][5] for i in ids}

    schedules = []
    closing = original_close

    # Construir los recorridos diarios 
    for r, group in enumerate(groups, 1):
        for t in range(days):
            # Recorre cada ruta y cada día. Las rutas se numeran desde 1.
            remaining = {i for i in group if demand[i][t] > 0}
            previous, clock, visits = 0, opening, []
            while remaining:
                # Escoge el siguiente cliente que podria empezar antes. 
                # inicio posible = max{apertura del cliente, hora actual + tiempo de viaje}, si hay empate el identificador mas bajo gana.
                i = min(remaining, key=lambda j: (
                    max(ready[j], clock + distancia(rows, previous, j)), j
                ))
                # Calcula el inicio de servicio y actualiza la ventana de cierre si es necesario.
                start = max(ready[i], clock + distancia(rows, previous, i))
                due[i] = max(due[i], math.ceil(start))

                # Registra la visita, suma el servicio, actualiza la ubicación y elimina al cliente de los pendientes.
                visits.append((i, start))
                clock, previous = start + service[i], i
                remaining.remove(i)

            # Volver al deposito, calcula el regreso y amplía el cierre del depósito si hace falta.
            return_time = clock + distancia(rows, previous, 0)
            closing = max(closing, math.ceil(return_time))
            schedules.append((r, t, visits, return_time))

    # La siguiente parte recorre los horarios construidos y los comprueba.
    # Validacion independiente de la solución factible de referencia: cobertura, consistencia, carga y tiempos.
    for t in range(days):
        seen = []
        for r, day, visits, return_time in schedules:
            if day != t:
                continue
            previous, clock, load = 0, opening, 0
            for i, start in visits:
                if i not in groups[r-1] or demand[i][t] <= 0:
                    raise ValueError("solución factible de referencia inconsistente")
                if not ready[i] <= start <= due[i]:
                    raise ValueError("Ventana incumplida")
                if start + 1e-8 < clock + distancia(rows, previous, i):
                    raise ValueError("Tiempo de viaje incumplido")
                seen.append(i)
                load += demand[i][t]
                previous, clock = i, start + service[i]
            if load > q or return_time > closing:
                raise ValueError("Capacidad u horario incumplido")
            if abs(return_time - clock - distancia(rows, previous, 0)) > 1e-8:
                raise ValueError("Retorno inconsistente")
        expected = sorted(i for i in ids if demand[i][t] > 0)
        if sorted(seen) != expected:
            raise ValueError("Cobertura diaria incorrecta")

    destination.parent.mkdir(parents=True, exist_ok=True)
    name = destination.stem
    text = [f"NAME {name}", "FORMAT CONVRP_MULTIDAY_V1",
            f"SOURCE {source.name}",
            f"SEED {seed}", f"DAYS {days}", f"VEHICLES {vehicles}",
            f"CAPACITY {q}", "DISTANCE EUCLIDEAN_DOUBLE",
            "TRAVEL_TIME EQUAL_DISTANCE", "TIME_UNIT ABSTRACT",
            "", "DEPOT", "ID X Y OPEN CLOSE",
            f"0 {rows[0][1]:g} {rows[0][2]:g} {opening:g} {closing:g}",
            "", "CUSTOMERS", "ID X Y READY DUE SERVICE"]
    for i in ids:
        text.append(f"{i} {rows[i][1]:g} {rows[i][2]:g} "
                    f"{ready[i]:g} {due[i]:g} {service[i]:g}")
    text += ["", "DAILY_DEMAND", "ID " + " ".join(f"DAY{t+1}" for t in range(days))]
    text += [f"{i} " + " ".join(map(str, demand[i])) for i in ids]
    text += ["", "EOF"]
    destination.write_text("\n".join(text) + "\n", encoding="utf-8")

    widened = [i for i in ids if due[i] != rows[i][5]]
    report = [f"Instancia: {name}", "solución factible de referencia VERIFICADO (no es una solucion optima)",
              "Dificultad computacional: PENDIENTE de medir con el solver.",
              f"Vehiculos originales: {original_vehicles}; nuevos: {vehicles}",
              f"Capacidad original: {original_q:g}; nueva: {q}; margen: {slack:.1%}",
              f"Cierre original: {original_close:g}; nuevo: {closing:g}",
              f"Ventanas con limite superior ampliado: {len(widened)}", ""]
    for i in widened:
        report.append(f"Cliente {i}: DUE {rows[i][5]:g} -> {due[i]:g}")
    for r, group in enumerate(groups, 1):
        report += [f"\nAsignacion solución factible de referencia ruta {r}: {group}", f"Cargas: {loads[r-1]}"]
    for r, t, visits, return_time in schedules:
        route = " -> ".join(["0"] + [str(i) for i, _ in visits] + ["0"])
        starts = ", ".join(f"{i}: {w:.6f}" for i, w in visits)
        report += [f"\nRuta {r}, dia {t+1}: {route}",
                   f"Salida fija: {opening:g}; retorno: {return_time:.6f}",
                   f"Inicio de servicio (valores mostrados redondeados): {starts}"]
    destination.with_name(name + "_verificacion.txt").write_text(
        "\n".join(report) + "\n", encoding="utf-8")
    print(f"{destination.name}: {n} clientes, {days} dias, {vehicles} vehiculos, "
          f"Q={q}, cierre={closing:g}, ventanas ampliadas={len(widened)}; solución factible de referencia OK")

In [ ]:
ARCHIVO_I1 = PATH_GENERADAS / "I1.txt"
ARCHIVO_I2 = PATH_GENERADAS / "I2.txt"

generar(
    source=ARCHIVO_R108,
    destination=ARCHIVO_I1,
    n=20,
    days=3,
    vehicles=3,
    seed=42,
    slack=0.05,
)

generar(
    source=ARCHIVO_R110,
    destination=ARCHIVO_I2,
    n=30,
    days=3,
    vehicles=3,
    seed=43,
    slack=0.05,
)

# Sección 3 Lector de instancias propias

In [ ]:
def leer_multidia(path):
    """Lee el formato CONVRP_MULTIDAY_V1 generado por nuestro código."""
    path = Path(path)

    cabecera = {}
    clientes = {}
    demandas = {}
    deposito = None
    seccion = None

    for numero_linea, linea in enumerate(
        path.read_text(encoding="utf-8-sig").splitlines(), start=1
    ):
        linea = linea.strip()

        if not linea or linea.startswith("#"):
            continue

        if linea == "EOF":
            break

        if linea in {"DEPOT", "CUSTOMERS", "DAILY_DEMAND"}:
            seccion = linea
            continue

        partes = linea.split()

        # Ignorar títulos de columnas.
        if partes[0] == "ID":
            continue

        try:
            if seccion is None:
                clave, valor = linea.split(maxsplit=1)
                cabecera[clave] = valor

            elif seccion == "DEPOT":
                if len(partes) != 5 or deposito is not None:
                    raise ValueError("Debe existir una única fila de depósito")

                identificador = int(partes[0])
                x, y, apertura, cierre = map(float, partes[1:])

                deposito = {
                    "id": identificador,
                    "x": x,
                    "y": y,
                    "apertura": apertura,
                    "cierre": cierre,
                }

            elif seccion == "CUSTOMERS":
                if len(partes) != 6:
                    raise ValueError("Fila de cliente con columnas incorrectas")

                i = int(partes[0])

                if i in clientes:
                    raise ValueError(f"Cliente duplicado: {i}")

                x, y, inicio, fin, servicio = map(float, partes[1:])

                clientes[i] = {
                    "x": x,
                    "y": y,
                    "inicio": inicio,
                    "fin": fin,
                    "servicio": servicio,
                }

            elif seccion == "DAILY_DEMAND":
                i = int(partes[0])

                if i in demandas:
                    raise ValueError(f"Demanda duplicada para cliente {i}")

                demandas[i] = list(map(float, partes[1:]))

        except ValueError as error:
            raise ValueError(
                f"{path.name}, línea {numero_linea}: {error}"
            ) from error

    # Comprobaciones de estructura.
    if cabecera.get("FORMAT") != "CONVRP_MULTIDAY_V1":
        raise ValueError("Formato multidía no reconocido")

    dias = int(cabecera["DAYS"])
    vehiculos = int(cabecera["VEHICLES"])
    capacidad = float(cabecera["CAPACITY"])

    if dias <= 0 or vehiculos <= 0 or capacidad <= 0:
        raise ValueError("Días, vehículos y capacidad deben ser positivos")

    if deposito is None or deposito["id"] != 0:
        raise ValueError("Falta el depósito con identificador 0")

    if not 0 <= deposito["apertura"] <= deposito["cierre"]:
        raise ValueError("Horario del depósito inválido")

    n = len(clientes)

    # Nuestros archivos generados seleccionan los clientes 1,...,n.
    if n == 0 or set(clientes) != set(range(1, n + 1)):
        raise ValueError("Se esperan clientes numerados desde 1 hasta n")

    if set(demandas) != set(clientes):
        raise ValueError("Los clientes y las filas de demanda no coinciden")

    for i in clientes:
        c = clientes[i]

        if not 0 <= c["inicio"] <= c["fin"]:
            raise ValueError(f"Ventana inválida del cliente {i}")

        if c["servicio"] <= 0:
            raise ValueError(f"Servicio no positivo del cliente {i}")

        if len(demandas[i]) != dias:
            raise ValueError(f"Cantidad de días incorrecta del cliente {i}")

        if any(q < 0 for q in demandas[i]):
            raise ValueError(f"Demanda negativa del cliente {i}")

    if (
        cabecera.get("DISTANCE") != "EUCLIDEAN_DOUBLE"
        or cabecera.get("TRAVEL_TIME") != "EQUAL_DISTANCE"
    ):
        raise ValueError("Convención de distancia o tiempo no compatible")

    # Conjuntos del modelo.
    C = list(range(1, n + 1))
    D = list(range(1, dias + 1))
    R = list(range(1, vehiculos + 1))

    retorno = n + 1
    N = [0] + C + [retorno]

    # Todos los clientes tienen ventana en nuestro formato actual.
    N_TW = C.copy()

    # Depósito de salida y retorno: misma ubicación física.
    coordenadas = {
        0: (deposito["x"], deposito["y"]),
        retorno: (deposito["x"], deposito["y"]),
    }
    coordenadas.update({
        i: (clientes[i]["x"], clientes[i]["y"]) for i in C
    })

    td = {
        (i, j): math.hypot(
            coordenadas[i][0] - coordenadas[j][0],
            coordenadas[i][1] - coordenadas[j][1],
        )
        for i in N
        for j in N
    }

    tt = td.copy()

    q = {
        (i, t): demandas[i][t - 1]
        for i in C
        for t in D
    }

    O = [(i, t) for i in C for t in D if q[i, t] > 0]

    st = {
        (i, t): clientes[i]["servicio"]
        for i, t in O
    }

    # Arcos diarios: solo clientes con pedido ese día.
    A = []

    for t in D:
        activos = [i for i in C if q[i, t] > 0]

        A.append((0, retorno, t))

        A.extend((0, i, t) for i in activos)
        A.extend((i, retorno, t) for i in activos)

        A.extend(
            (i, j, t)
            for i in activos
            for j in activos
            if i != j
        )

    return {
        "nombre": cabecera.get("NAME", path.stem),
        "C": C,
        "D": D,
        "R": R,
        "N": N,
        "N_TW": N_TW,
        "O": O,
        "A": A,
        "retorno": retorno,
        "coordenadas": coordenadas,
        "td": td,
        "tt": tt,
        "q": q,
        "st": st,
        "a": {i: clientes[i]["inicio"] for i in C},
        "b": {i: clientes[i]["fin"] for i in C},
        "Q": capacidad,
        "aW": deposito["apertura"],
        "bW": deposito["cierre"],
    }

In [ ]:
# Leer desde la carpeta de instancias generadas.
instancia_1 = leer_multidia(PATH_GENERADAS / "I1.txt")
instancia_2 = leer_multidia(PATH_GENERADAS / "I2.txt")

instancias = {
    "I1": instancia_1,
    "I2": instancia_2,
}

for identificador, datos in instancias.items():
    print(
        f"{identificador}: "
        f"{len(datos['C'])} clientes, "
        f"{len(datos['D'])} días, "
        f"{len(datos['R'])} vehículos, "
        f"capacidad {datos['Q']}"
    )

# Sección 4 Solver Gurobi (modelo MIP)

Formulación del **Ruteo Consistente de Vehículos multidía** como programa entero mixto.

**Conjuntos:** clientes $C$, días $D$, vehículos $R$, arcos diarios $A$, y $O = \{(i,t) : q_{it} > 0\}$ (pedidos obligatorios).

**Variables:**

| Var | Significado |
|-----|-------------|
| $x_{ijtr} \in \{0,1\}$ | el vehículo $r$ viaja de $i$ a $j$ el día $t$ |
| $v_{itr} \in \{0,1\}$ | el vehículo $r$ atiende al cliente $i$ el día $t$ |
| $z_{ir} \in \{0,1\}$ | el cliente $i$ queda asignado al vehículo $r$ (consistencia) |
| $s_{it} \ge 0$ | inicio de servicio de $i$ el día $t$ |
| $u_{it} \ge 0$ | carga acumulada al llegar a $i$ el día $t$ |

**Restricciones:**

1. **Flujo** por día y vehículo: a lo sumo una salida del depósito; entra = sale = $v_{itr}$ en cada cliente; los arcos a retorno = los arcos de salida.
2. **Cobertura:** $\sum_r v_{itr} = 1$ para todo $(i,t) \in O$.
3. **Capacidad:** $\sum_i q_{it} v_{itr} \le Q$ por (día, vehículo).
4. **Carga acumulada (MTZ):** $u_{jt} \ge u_{it} + q_{jt} - Q(1 - \sum_r x_{ijtr})$; además elimina subrutas.
5. **Ventanas de tiempo:** llegada al depósito $\ge$ apertura + viaje; secuencia $s_{jt} \ge s_{it} + \text{servicio}_i + tt_{ij} - M(1-x_{ijtr})$; retorno $\le$ cierre del depósito.
6. **Consistencia:** $\sum_r z_{ir} = 1$ y $v_{itr} \le z_{ir}$ → el mismo vehículo atiende a $i$ **todos** los días que tiene pedido.
7. **Simetría:** los vehículos son idénticos, se ordenan por número de visitas ($\sum_{it} v_{itr} \ge \sum_{it} v_{ir+1,t}$).

**Objetivo:** minimizar la distancia total recorrida en todos los días.

Se usa la **solución factible de referencia** (Sección 2, archivo `*_verificacion.txt`) como *MIP start*: Gurobi arranca ya con una solución factible y solo busca mejorarla.


In [ ]:
# ============================================================
# Seccion 4.1: construccion del modelo MIP con Gurobi
# ============================================================
import re
import time

import gurobipy as gp
from gurobipy import GRB


def construir_modelo(instancia, tiempo_limite=300, gap_objetivo=0.01,
                     semilla=42, verbo=True):
    """Devuelve (modelo, variables, datos_auxiliares)."""
    C, D, R = instancia["C"], instancia["D"], instancia["R"]
    ret, A, O = instancia["retorno"], instancia["A"], set(instancia["O"])
    q, st, td, tt = (instancia[k] for k in ("q", "st", "td", "tt"))
    a, b, aW, bW, Q = (instancia[k] for k in ("a", "b", "aW", "bW", "Q"))
    activos = {t: [i for i in C if (i, t) in O] for t in D}

    m = gp.Model(f"ConVRP_{instancia['nombre']}")
    m.Params.OutputFlag = 1 if verbo else 0
    m.Params.TimeLimit = tiempo_limite
    m.Params.MIPGap = gap_objetivo
    m.Params.Seed = semilla

    # --- Variables ---
    x = {(i, j, t, r): m.addVar(vtype=GRB.BINARY, name=f"x_{i}_{j}_d{t}_r{r}")
         for (i, j, t) in A for r in R}          # arco (i,j) dia t, vehiculo r
    v = {(i, t, r): m.addVar(vtype=GRB.BINARY, name=f"v_{i}_d{t}_r{r}")
         for (i, t) in O for r in R}             # i atendido por r el dia t
    z = {(i, r): m.addVar(vtype=GRB.BINARY, name=f"z_{i}_r{r}")
         for i in C for r in R}                  # i asignado a r (consistencia)
    s = {(i, t): m.addVar(lb=a[i], ub=b[i], name=f"s_{i}_d{t}")
         for (i, t) in O}                        # inicio de servicio
    u = {(i, t): m.addVar(lb=0, ub=Q, name=f"u_{i}_d{t}")
         for (i, t) in O}                        # carga acumulada

    # --- Objetivo: minimizar distancia total ---
    m.setObjective(
        gp.quicksum(td[i, j] * x[i, j, t, r] for (i, j, t) in A for r in R),
        GRB.MINIMIZE)

    # --- (1) Flujos por dia y vehiculo ---
    for t in D:
        for r in R:
            m.addConstr(
                gp.quicksum(x[0, j, t, r] for j in activos[t] + [ret]) <= 1,
                name=f"salida_d{t}_r{r}")
            for i in activos[t]:
                m.addConstr(
                    gp.quicksum(x[i, j, t, r] for j in activos[t] + [ret] if j != i)
                    == v[i, t, r], name=f"flujo_out_{i}_d{t}_r{r}")
                m.addConstr(
                    gp.quicksum(x[j, i, t, r] for j in activos[t] + [0] if j != i)
                    == v[i, t, r], name=f"flujo_in_{i}_d{t}_r{r}")
            m.addConstr(
                gp.quicksum(x[i, ret, t, r] for i in activos[t])
                == gp.quicksum(x[0, j, t, r] for j in activos[t]),
                name=f"retorno_d{t}_r{r}")

    # --- (2) Cada pedido atendido exactamente un vehiculo ---
    for (i, t) in O:
        m.addConstr(gp.quicksum(v[i, t, r] for r in R) == 1,
                    name=f"cobertura_{i}_d{t}")

    # --- (3) Capacidad por vehiculo y dia ---
    for t in D:
        for r in R:
            m.addConstr(gp.quicksum(q[i, t] * v[i, t, r] for i in activos[t]) <= Q,
                        name=f"capacidad_d{t}_r{r}")

    # --- (4) Carga acumulada: refuerza capacidad y elimina subrutas ---
    for (i, j, t) in A:
        if i == 0 or j == ret:
            continue
        m.addConstr(u[j, t] >= u[i, t] + q[j, t]
                    - Q * (1 - gp.quicksum(x[i, j, t, r] for r in R)),
                    name=f"carga_{i}_{j}_d{t}")
    for (i, t) in O:
        m.addConstr(u[i, t] >= q[i, t], name=f"carga_min_{i}_d{t}")

    # --- (5) Ventanas de tiempo ---
    for (i, t) in O:
        Mo = bW + tt[0, i] - a[i]
        m.addConstr(s[i, t] >= aW + tt[0, i]
                    - Mo * (1 - gp.quicksum(x[0, i, t, r] for r in R)),
                    name=f"salida_dep_{i}_d{t}")
        Mi = bW + st[i, t] + tt[i, ret] - a[i]
        m.addConstr(s[i, t] + st[i, t] + tt[i, ret]
                    <= bW + Mi * (1 - gp.quicksum(v[i, t, r] for r in R)),
                    name=f"retorno_dep_{i}_d{t}")
    for (i, j, t) in A:
        if i == 0 or j == ret:
            continue
        for r in R:
            Mij = b[i] + st[i, t] + tt[i, j] - a[j]
            m.addConstr(s[j, t] >= s[i, t] + st[i, t] + tt[i, j]
                        - Mij * (1 - x[i, j, t, r]),
                        name=f"secuencia_{i}_{j}_d{t}_r{r}")

    # --- (6) Consistencia: mismo vehiculo en todos los dias que atiende a i ---
    for i in C:
        m.addConstr(gp.quicksum(z[i, r] for r in R) == 1,
                    name=f"un_vehiculo_{i}")
        for (j, t) in O:
            if j == i:
                for r in R:
                    m.addConstr(v[i, t, r] <= z[i, r],
                                name=f"consistencia_{i}_d{t}_r{r}")

    # --- (7) Romper simetria: vehiculos identicos ordenados por visitas ---
    visitas = {r: gp.quicksum(v[i, t, r] for (i, t) in O) for r in R}
    for r in R[:-1]:
        m.addConstr(visitas[r] >= visitas[r + 1], name=f"simetria_r{r}")

    datos = {"activos": activos, "q": q, "st": st, "td": td, "tt": tt,
             "a": a, "b": b, "aW": aW, "bW": bW, "Q": Q}
    return m, {"x": x, "v": v, "z": z, "s": s, "u": u}, datos


In [ ]:
# ============================================================
# Seccion 4.2: MIP start desde la solucion de referencia (verificacion)
# ============================================================
def solucion_de_verificacion(path):
    """Extrae rutas y inicios de servicio del archivo *_verificacion.txt."""
    lineas = Path(path).read_text(encoding="utf-8").splitlines()
    rutas, inicios = {}, {}
    patron_ruta = re.compile(r"^Ruta (\d+), dia (\d+): (.+)$")
    patron_inicio = re.compile(r"^Inicio de servicio .*\): (.*)$")
    for linea in lineas:
        linea = linea.strip()
        mr = patron_ruta.match(linea)
        if mr:
            r, t = int(mr.group(1)), int(mr.group(2))
            nodos = [int(x) for x in mr.group(3).split("->")]
            rutas[r, t] = [i for i in nodos[1:-1]]   # sin el depot
            continue
        mi = patron_inicio.match(linea)
        if mi and rutas:
            r, t = list(rutas)[-1]                   # ultima ruta leida
            for par in mi.group(1).split(","):
                i, val = par.split(":")
                inicios[int(i), t] = float(val)
    return rutas, inicios


def aplicar_inicio(vars_, instancia, rutas):
    """Fija todas las variables a la solucion de referencia (MIP start)."""
    C, D, R = instancia["C"], instancia["D"], instancia["R"]
    ret, A, O = instancia["retorno"], instancia["A"], set(instancia["O"])
    q, st, tt = instancia["q"], instancia["st"], instancia["tt"]
    a, aW = instancia["a"], instancia["aW"]
    x, v, z, s, u = vars_["x"], vars_["v"], vars_["z"], vars_["s"], vars_["u"]

    # Los vehiculos son identicos: permutarlos para cumplir la restriccion de
    # simetria (visitas en orden descendente) no cambia la factibilidad.
    visitas = {r: sum(len(rutas.get((r, t), [])) for t in D) for r in R}
    orden = sorted(R, key=lambda r: -visitas[r])
    remap = {viejo: nuevo for nuevo, viejo in enumerate(orden, start=R[0])}
    rutas = {(remap[r], t): cli for (r, t), cli in rutas.items()}

    pertenece = {}
    for (r, t), clientes in rutas.items():
        for i in clientes:
            pertenece.setdefault(i, r)

    # x en 0 por defecto (evita que Gurobi reciba un start parcial)
    for clave in x:
        x[clave].Start = 0

    for t in D:
        act = {i for i in C if (i, t) in O}
        for r in R:
            ruta = rutas.get((r, t), [])
            vistos = set(ruta)
            for i in act:
                v[i, t, r].Start = 1 if i in vistos else 0
            # arcos: depot -> primero, entre consecutivos, ultimo -> retorno
            for p, j in zip([0] + ruta, ruta + [ret]):
                x[p, j, t, r].Start = 1
            # inicio de servicio por propagacion exacta (sin redondeos) y carga
            reloj, anterior, carga = aW, 0, 0.0
            for i in ruta:
                inicio = max(a[i], reloj + tt[anterior, i])
                carga += q[i, t]
                if (i, t) in s:
                    s[i, t].Start = inicio
                    u[i, t].Start = carga
                reloj = inicio + st[i, t]
                anterior = i

    for i in C:
        r = pertenece[i]
        for r2 in R:
            z[i, r2].Start = 1 if r2 == r else 0


In [ ]:
# ============================================================
# Seccion 4.3: validacion independiente de la solucion del solver
# ============================================================
def validar_solucion(instancia, vars_):
    """Revisa cobertura, capacidad, ventanas, tiempos, retorno y consistencia.
    Devuelve (rutas, distancia_total). Lanza AssertionError si algo falla."""
    C, D, R = instancia["C"], instancia["D"], instancia["R"]
    ret, A, O = instancia["retorno"], instancia["A"], set(instancia["O"])
    q, st, tt = instancia["q"], instancia["st"], instancia["tt"]
    a, b, aW, bW, Q = (instancia[k] for k in ("a", "b", "aW", "bW", "Q"))
    x, v, s = vars_["x"], vars_["v"], vars_["s"]

    vehiculo_de, rutas = {}, {}
    for t in D:
        act = {i for i in C if (i, t) in O}
        atendidos = set()
        for r in R:
            arcos = [(i, j) for (i, j, dd) in A if dd == t and x[i, j, dd, r].X > 0.5]
            succ = dict(arcos)
            if not succ:
                continue
            ruta, nodo = [], 0
            while succ.get(nodo, ret) != ret:
                nodo = succ[nodo]
                assert nodo in act and nodo not in ruta, "subruta o nodo repetido"
                ruta.append(nodo)
            rutas[r, t] = ruta
            carga = sum(q[i, t] for i in ruta)
            assert carga <= Q, f"capacidad dia {t} vehiculo {r}: {carga} > {Q}"
            reloj, anterior = aW, 0
            for i in ruta:
                inicio = s[i, t].X
                assert a[i] - 1e-6 <= inicio <= b[i] + 1e-6, \
                    f"ventana cliente {i} dia {t}"
                assert inicio >= reloj + tt[anterior, i] - 1e-6, \
                    f"tiempo de viaje violado en {i} dia {t}"
                reloj = inicio + st[i, t]
                anterior = i
                atendidos.add(i)
                # consistencia: mismo vehiculo en todos los dias
                if i in vehiculo_de:
                    assert vehiculo_de[i] == r, f"vehiculo inconsistente en {i}"
                else:
                    vehiculo_de[i] = r
            retorno = reloj + tt[anterior, ret]
            assert retorno <= bW + 1e-6, \
                f"retorno tardio dia {t} vehiculo {r}: {retorno}"
        assert atendidos == act, f"cobertura incorrecta dia {t}: {atendidos ^ act}"

    dist = sum(instancia["td"][i, j] * x[i, j, t, r].X
               for (i, j, t) in instancia["A"] for r in R)
    return rutas, dist


# Sección 5 Resolución y validación


In [ ]:
# ============================================================
# Seccion 4.4: resolver una instancia y reportar
# ============================================================
def resolver(nombre, tiempo_limite=300, gap_objetivo=0.01, semilla=42,
             verbo=True, guardar=True):
    inst = instancias[nombre]
    m, vars_, _ = construir_modelo(inst, tiempo_limite=tiempo_limite,
                                   gap_objetivo=gap_objetivo, semilla=semilla,
                                   verbo=verbo)

    # MIP start con la solucion de referencia
    archivo_verif = PATH_GENERADAS / f"{nombre}_verificacion.txt"
    if archivo_verif.is_file():
        rutas_ref, _ = solucion_de_verificacion(archivo_verif)
        aplicar_inicio(vars_, inst, rutas_ref)
        m.update()
        print(f"[{nombre}] MIP start cargado desde {archivo_verif.name}")

    t0 = time.time()
    m.optimize()
    tiempo = time.time() - t0

    estados = {1: "OPTIMO", 2: "OPTIMO", 3: "INFACTIBLE", 9: "TIME LIMIT",
               13: "INTERRUPTADO"}
    print(f"\n[{nombre}] estado={estados.get(m.Status, m.Status)} | "
          f"tiempo={tiempo:.1f}s | objetivos soluciones={m.SolCount}")
    if m.SolCount == 0:
        print(f"[{nombre}] Sin solucion factible en {tiempo_limite}s")
        return None

    print(f"[{nombre}] distancia={m.ObjVal:.2f} | gap={m.MIPGap:.2%} | "
          f"mejor cota={m.ObjBound:.2f}")
    rutas, dist = validar_solucion(inst, vars_)
    print(f"[{nombre}] SOLUCION VALIDADA por validador independiente")

    lineas = [f"Instancia {nombre}",
              f"distancia={dist:.2f} gap={m.MIPGap:.4%} "
              f"cota={m.ObjBound:.2f} tiempo={tiempo:.1f}s "
              f"estado={estados.get(m.Status, m.Status)}", ""]
    for t in inst["D"]:
        for r in inst["R"]:
            ruta = rutas.get((r, t))
            if not ruta:
                continue
            carga = sum(inst["q"][i, t] for i in ruta)
            inicios = ", ".join(f"{i}:{vars_['s'][i, t].X:.2f}" for i in ruta)
            linea = (f"dia {t} veh {r}: 0 -> {' -> '.join(map(str, ruta))} -> 0 | "
                     f"carga {carga:g}/{inst['Q']:g} | inicio {inicios}")
            print("  ", linea)
            lineas.append(linea)

    if guardar:
        salida = Path("results") / f"{nombre}_solver.txt"
        salida.parent.mkdir(parents=True, exist_ok=True)
        salida.write_text("\n".join(lineas) + "\n", encoding="utf-8")
        print(f"[{nombre}] reporte guardado en {salida}")

    return {"instancia": inst, "vars": vars_, "model": m, "rutas": rutas,
            "distancia": dist, "tiempo": tiempo, "gap": m.MIPGap}


In [ ]:
# ============================================================
# Seccion 4.5: resolver ambas instancias
# ============================================================
resultado_i1 = resolver("I1", tiempo_limite=300, gap_objetivo=0.01)
resultado_i2 = resolver("I2", tiempo_limite=300, gap_objetivo=0.01)


# Sección 6: ajuste automático y sensibilidad de capacidad
Puntos 3 y 4. La siguiente celda puede ejecutarse sola desde la carpeta del proyecto.
Requiere licencia para las instancias completas. Ejecuta comparaciones nuevas con un hilo y 600 s;
no reutiliza las corridas de 300 s de la sección anterior. Ver `README.md`.

In [ ]:
from analisis_34 import cargar_funciones, ejecutar_instancia, ROOT

api_34 = cargar_funciones()
for nombre_34 in ("I1", "I2"):
    ejecutar_instancia(nombre_34, api_34, ROOT / "results" / "analisis_34" / nombre_34, 1800)
